**1. Import Libraries**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import random

**ADD MOBILENETV1+BIGRU MODEL CODE**

In [ ]:
# just run the data preprocessing & the model definition, dont run the model.fit part, that is just for reference

**4. Create Federated Clients**

In [ ]:
NUM_CLIENTS = 5

client_data = {}

X_split = np.array_split(X_train, NUM_CLIENTS)
y_split = np.array_split(y_train, NUM_CLIENTS)

for i in range(NUM_CLIENTS):
    client_data[i] = (X_split[i], y_split[i])

**5. Client Local Training Function**

In [ ]:
def client_update(global_weights, dataset, local_epochs=2):

    local_model = MobileNetV1_BiGRU()

    local_model.set_weights(global_weights)

    X, y = dataset

    local_model.fit(
        X,
        y,
        epochs=local_epochs,
        batch_size=32,
        verbose=0
    )

    return local_model.get_weights()

**6. Federated Averaging (FedAvg)**

In [ ]:
def federated_average(client_weights):

    avg_weights = []

    for weights in zip(*client_weights):
        avg_weights.append(np.mean(weights, axis=0))

    return avg_weights

**7. Federated Training Loop (Server)**

In [ ]:
ROUNDS = 30
CLIENTS_PER_ROUND = 5

global_model = MobileNetV1_BiGRU()

for round_num in range(ROUNDS):

    print("FL Round:", round_num)

    global_weights = global_model.get_weights()

    client_weights = []

    # randomly choose clients
    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:

        weights = client_update(
            global_weights,
            client_data[client]
        )

        client_weights.append(weights)

    # server aggregation
    new_global_weights = federated_average(client_weights)

    global_model.set_weights(new_global_weights)

**8. Evaluate Global Model**

In [ ]:
loss, accuracy = global_model.evaluate(X_test, y_test)
print("Final Global Model Accuracy:", accuracy)